# Komparasi Model — Analisis Sentimen MBG

**Notebook:** `06_model_comparison.ipynb`  
**Peneliti:** Khoeru Roziqin  
**Tanggal:** 04-05-26  

---

## Deskripsi

Notebook ini mengompilasi dan memvisualisasikan hasil evaluasi **seluruh model** yang telah dilatih pada notebook 03–05 untuk membentuk tabel komparasi komprehensif.

### Model yang Dibandingkan

| # | Model | Tipe | Feature | Notebook |
|---|---|---|---|---|
| 1 | **SVM** | ML Baseline | TF-IDF | 05 |
| 2 | **Logistic Regression** | ML Baseline | TF-IDF | 05 |
| 3 | **Random Forest** | ML Baseline | TF-IDF | 05 |
| 4 | **IndoBERT-only** | DL Baseline | IndoBERT [CLS] | 04 |
| 5 | **IndoBERT+CNN** | **Proposed Model** | IndoBERT + CNN 1D | 03 |

### Semua model menggunakan:
- **Split data identik** — test_set_indices.npy yang sama (1.329 test, 5.313 train+val)
- **Metrik identik** — F1-Macro sebagai metrik utama
- **Kondisi data identik** — S1 (class weighting)

---

## Hasil Sementara (dari log notebook 03–05)

| Model | F1-Macro | Accuracy |
|---|---|---|
| SVM | 0.8267 | 0.8322 |
| Logistic Regression | 0.8251 | 0.8284 |
| Random Forest | 0.7966 | 0.8006 |
| IndoBERT-only | 0.8497 | 0.8570 |
| **IndoBERT+CNN** | **0.8453** | **0.8495** |

## 0. Setup

> Notebook ini **tidak membutuhkan GPU** — hanya memuat CSV hasil dan membuat visualisasi.  
> Default: **Google Colab**. Untuk Kaggle: uncomment sel Kaggle.

In [ ]:
import sys, os

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_KAGGLE:
    print('Environment: Kaggle Notebook')
elif IS_COLAB:
    print('Environment: Google Colab')
else:
    print('Environment: Local / Unknown')

In [ ]:
# ══════════════════════════════════════════════════════
#  SETUP — KAGGLE NOTEBOOK
# ══════════════════════════════════════════════════════

import os
print('✅ Setup Kaggle selesai. (CPU)')

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix
warnings.filterwarnings('ignore')

print('✅ Library berhasil dimuat.')

## 1. Konfigurasi Path

Notebook ini membaca output dari notebook 03, 04, dan 05.

In [ ]:
# ── PATH — KAGGLE ────────
BASE_DIR      = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'
DIR_NB03      = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'
DIR_NB04      = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'
DIR_NB05      = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'
OUTPUT_DIR    = '/kaggle/working/comparison_results'
OUTPUTS_INPUT = '/kaggle/input/datasets/khoeruroziqin/mbg-training-outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Restore semua output dari dataset (Kaggle only) ───────────────────────────
import shutil, os

_files_to_restore = [
    # Dari notebook 03
    'phase3_kfold_results.csv',
    'test_predictions.csv',
    # Dari notebook 04
    'bert_only_summary.csv',
    'bert_only_test_predictions.csv',
    'bert_only_fold_details.csv',
    # Dari notebook 05
    'ml_baseline_summary.csv',
    'ml_baseline_fold_details.csv',
    'svm_test_predictions.csv',
    'logisticregression_test_predictions.csv',
    'randomforest_test_predictions.csv',
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Restoring files dari mbg-training-outputs...')
for fname in _files_to_restore:
    src = f'{OUTPUTS_INPUT}/{fname}'
    dst = f'{OUTPUT_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  ✔ {fname}')
    else:
        print(f'  ⚠️  Tidak ditemukan: {fname}')
print('Done.')

# ── Konstanta ─────────────────────────────────────────────────────────────────
LABEL_NAMES = ['positive', 'negative', 'neutral']
LABEL2ID    = {'positive': 0, 'negative': 1, 'neutral': 2}

# Hasil aktual dari notebook 03, 04, 05 (update jika ada perubahan)
PROPOSED_F1  = 0.8453
PROPOSED_ACC = 0.8495

print('✅ Konfigurasi berhasil dimuat.')
print(f'   NB03 output : {DIR_NB03}')
print(f'   NB04 output : {DIR_NB04}')
print(f'   NB05 output : {DIR_NB05}')
print(f'   Output dir  : {OUTPUT_DIR}')

## 2. Kompilasi Hasil Semua Model

Mengumpulkan semua metrik dari CSV output notebook 03–05 ke dalam satu DataFrame terpadu.

In [ ]:
# ── Load hasil notebook 05 — ML Baseline ──────────────────────────────────────
df_ml = pd.read_csv(f'{DIR_NB05}/ml_baseline_summary.csv')
print(f'ML baseline dimuat: {len(df_ml)} model')
print(f'  Kolom: {list(df_ml.columns)}')

# ── Load hasil notebook 04 — BERT-only Baseline ───────────────────────────────
df_bert = pd.read_csv(f'{DIR_NB04}/bert_only_summary.csv')
print(f'BERT-only baseline dimuat: {len(df_bert)} model')
print(f'  Kolom: {list(df_bert.columns)}')

# ── Load Phase 3 K-Fold untuk proposed model ──────────────────────────────────
df_p3 = pd.read_csv(f'{DIR_NB03}/phase3_kfold_results.csv')
s1    = df_p3[df_p3['condition']=='S1'].iloc[0]

# ── Susun proposed model row ──────────────────────────────────────────────────
proposed_row = {
    'model'                  : 'IndoBERT+CNN (Proposed)',
    'data_condition'         : 'S1',
    'kfold_f1_macro_mean'    : round(s1['f1_macro'], 4),
    'kfold_f1_macro_std'     : round(s1['f1_macro_std'], 4),
    'test_accuracy'          : 0.8495,
    'test_precision'         : 0.8465,
    'test_recall'            : 0.8452,
    'test_f1_macro'          : 0.8453,
    'test_f1_weighted'       : 0.8499,
    'f1_positive'            : 0.8889,
    'f1_negative'            : 0.8501,
    'f1_neutral'             : 0.7970,
}

# ── Kolom minimal yang wajib ada di semua model ───────────────────────────────
REQUIRED_COLS = [
    'model', 'data_condition',
    'kfold_f1_macro_mean', 'kfold_f1_macro_std',
    'test_accuracy', 'test_precision', 'test_recall',
    'test_f1_macro', 'test_f1_weighted',
    'f1_positive', 'f1_negative', 'f1_neutral',
]

def align_df(df, required_cols):
    """Pastikan df memiliki semua kolom yang dibutuhkan.
    Kolom yang tidak ada diisi NaN, kolom ekstra diabaikan."""
    for col in required_cols:
        if col not in df.columns:
            df[col] = None
            print(f'  ⚠️  Kolom "{col}" tidak ada di {df["model"].iloc[0] if "model" in df.columns else "?"} — diisi NaN')
    return df[required_cols]

df_ml_aligned   = align_df(df_ml.copy(),   REQUIRED_COLS)
df_bert_aligned = align_df(df_bert.copy(), REQUIRED_COLS)
df_proposed     = pd.DataFrame([{c: proposed_row.get(c, None) for c in REQUIRED_COLS}])

# ── Gabungkan semua model ─────────────────────────────────────────────────────
df_all = pd.concat([
    df_ml_aligned,
    df_bert_aligned,
    df_proposed
], ignore_index=True)

# Tambahkan kolom kategori
def get_category(model_name):
    if 'CNN' in model_name or 'Proposed' in model_name: return 'Proposed'
    if 'IndoBERT' in model_name or 'BERT' in model_name: return 'DL Baseline'
    return 'ML Baseline'

df_all['category'] = df_all['model'].apply(get_category)
df_all['rank']     = df_all['test_f1_macro'].rank(ascending=False).astype(int)
df_all = df_all.sort_values('test_f1_macro', ascending=False).reset_index(drop=True)

df_all.to_csv(f'{OUTPUT_DIR}/model_comparison_full.csv', index=False)

print(f'\n✅ Kompilasi selesai: {len(df_all)} model')
print(f'\nRankings berdasarkan F1-Macro test set:')
print(f' {"Rank":<5} {"Model":<30} {"Kategori":<15} {"F1-Macro":>10} {"Accuracy":>10}')
print('─' * 75)
for row in df_all.itertuples():
    marker = ' ← terbaik' if row.Index == 0 else ''
    print(f' {row.rank:<5} {row.model:<30} {row.category:<15} '
          f'{row.test_f1_macro:>10.4f} {row.test_accuracy:>10.4f}{marker}')

## 3. Tabel Komparasi Komprehensif

Tabel lengkap semua metrik untuk semua model — siap untuk Bab 4 laporan.

In [ ]:
# ── Tabel utama: semua metrik test set ───────────────────────────────────────
print('=' * 95)
print(' TABEL KOMPARASI MODEL — TEST SET (1.329 data)')
print('=' * 95)
print(f' {"#":<3} {"Model":<28} {"Kategori":<14} {"Acc":>8} {"Prec":>8} {"Rec":>8} '
      f'{"F1-Mac":>8} {"F1-Wgt":>8}')
print('─' * 95)
for i, row in enumerate(df_all.itertuples(), 1):
    marker = ' ★' if row.category == 'Proposed' else ''
    print(f' {i:<3} {row.model:<28} {row.category:<14} '
          f'{row.test_accuracy:>8.4f} {row.test_precision:>8.4f} '
          f'{row.test_recall:>8.4f} {row.test_f1_macro:>8.4f} '
          f'{row.test_f1_weighted:>8.4f}{marker}')
print('=' * 95)
print(' ★ = Proposed Model')

# ── Tabel per-class F1 ────────────────────────────────────────────────────────
print(f'\n{" Per-class F1-Score — Test Set":}')
print('=' * 75)
print(f' {"#":<3} {"Model":<28} {"Kategori":<14} {"Positive":>10} {"Negative":>10} {"Neutral":>10}')
print('─' * 75)
for i, row in enumerate(df_all.itertuples(), 1):
    print(f' {i:<3} {row.model:<28} {row.category:<14} '
          f'{row.f1_positive:>10.4f} {row.f1_negative:>10.4f} {row.f1_neutral:>10.4f}')
print('=' * 75)

# ── Tabel K-Fold validation ───────────────────────────────────────────────────
print(f'\n{" K-Fold Validation (5-Fold, Train+Val 5.313 data)":}')
print('=' * 75)
print(f' {"#":<3} {"Model":<28} {"Kategori":<14} {"F1-Mac Mean":>12} {"F1-Mac Std":>12}')
print('─' * 75)
for i, row in enumerate(df_all.itertuples(), 1):
    print(f' {i:<3} {row.model:<28} {row.category:<14} '
          f'{row.kfold_f1_macro_mean:>12.4f} {row.kfold_f1_macro_std:>12.4f}')
print('=' * 75)

# ── Analisis peningkatan proposed vs baseline ─────────────────────────────────
proposed = df_all[df_all['category']=='Proposed'].iloc[0]
best_dl   = df_all[df_all['category']=='DL Baseline'].iloc[0]
best_ml   = df_all[df_all['category']=='ML Baseline'].iloc[0]

print(f'\n Peningkatan Proposed Model vs Baseline:')
print(f'  vs {best_dl["model"]:<25}: Δ F1-Macro = {proposed["test_f1_macro"]-best_dl["test_f1_macro"]:+.4f}')
print(f'  vs {best_ml["model"]:<25}: Δ F1-Macro = {proposed["test_f1_macro"]-best_ml["test_f1_macro"]:+.4f}')

## 4. Visualisasi Komparasi

Menghasilkan visualisasi komprehensif untuk keperluan Bab 4 laporan.

In [ ]:
# Palet warna per kategori
CAT_COLORS = {
    'ML Baseline' : '#AFA9EC',
    'DL Baseline' : '#EF9F27',
    'Proposed'    : '#1D9E75',
}
bar_colors = [CAT_COLORS[c] for c in df_all['category']]
model_labels = [m.replace(' (Proposed)','').replace('IndoBERT','IndoBERT') for m in df_all['model']]

# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 1: F1-Macro Comparison (Bar Chart)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(range(len(df_all)), df_all['test_f1_macro'],
              color=bar_colors, edgecolor='none', alpha=0.92, width=0.6)

# Nilai di atas bar
for bar, val in zip(bars, df_all['test_f1_macro']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Garis referensi proposed model
proposed_f1_val = df_all[df_all['category']=='Proposed']['test_f1_macro'].values[0]
ax.axhline(y=proposed_f1_val, color='#1D9E75', linestyle='--',
           linewidth=1.2, alpha=0.6, label=f'Proposed F1={proposed_f1_val:.4f}')

ax.set_xticks(range(len(df_all)))
ax.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1-Macro (Test Set)', fontsize=11)
ax.set_title('Komparasi F1-Macro Semua Model — Test Set\n(diurutkan dari tertinggi)', fontsize=12)
ax.set_ylim(0.75, 0.91)
ax.legend(fontsize=9)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in CAT_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_f1_macro.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_f1_macro.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 2: Multi-metric Comparison (Grouped Bar)
# ══════════════════════════════════════════════════════════════════════════════
metrics_multi  = ['test_f1_macro','test_f1_weighted','test_accuracy','test_precision','test_recall']
labels_multi   = ['F1-Macro','F1-Weighted','Accuracy','Precision','Recall']
n_metrics      = len(metrics_multi)
n_models       = len(df_all)
x              = np.arange(n_metrics)
width          = 0.15

fig, ax = plt.subplots(figsize=(16, 6))
for mi, (row, color) in enumerate(zip(df_all.itertuples(), bar_colors)):
    offset = (mi - n_models/2 + 0.5) * width
    vals   = [getattr(row, m) for m in metrics_multi]
    ax.bar(x + offset, vals, width, label=row.model, color=color,
           edgecolor='none', alpha=0.90)

ax.set_xticks(x)
ax.set_xticklabels(labels_multi, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Komparasi Seluruh Metrik — Test Set', fontsize=12)
ax.set_ylim(0.72, 0.93)
ax.legend(fontsize=8, ncol=2, loc='lower right')
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_all_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_all_metrics.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 3: Per-class F1 Comparison
# ══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
cls_cols = ['f1_positive', 'f1_negative', 'f1_neutral']

for ci, (col, cls_name) in enumerate(zip(cls_cols, LABEL_NAMES)):
    ax = axes[ci]
    vals = df_all[col].values
    bars = ax.barh(range(len(df_all)), vals, color=bar_colors,
                   edgecolor='none', alpha=0.90, height=0.6)
    for bar, val in zip(bars, vals):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_yticks(range(len(df_all)))
    ax.set_yticklabels(model_labels, fontsize=9)
    ax.set_xlabel('F1-Score', fontsize=10)
    ax.set_title(f'F1-Score: {cls_name.capitalize()}', fontsize=11)
    ax.set_xlim(0.70, 0.96)
    ax.invert_yaxis()
    sns.despine(ax=ax)

plt.suptitle('Per-class F1-Score Semua Model — Test Set', fontsize=12, y=1.02)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in CAT_COLORS.items()]
fig.legend(handles=legend_patches, fontsize=9, loc='lower center',
           ncol=3, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_perclass_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_perclass_f1.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 4: K-Fold Stability Comparison (dengan error bar)
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(range(len(df_all)), df_all['kfold_f1_macro_mean'],
              yerr=df_all['kfold_f1_macro_std'],
              color=bar_colors, edgecolor='none', alpha=0.90, width=0.6,
              error_kw={'ecolor':'#555555', 'capsize':5, 'linewidth':1.2})

for bar, mean, std in zip(bars, df_all['kfold_f1_macro_mean'], df_all['kfold_f1_macro_std']):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.004,
            f'{mean:.4f}\n±{std:.4f}', ha='center', va='bottom',
            fontsize=8, fontweight='bold', linespacing=1.3)

ax.set_xticks(range(len(df_all)))
ax.set_xticklabels(model_labels, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1-Macro (mean ± std, K-Fold 5)', fontsize=11)
ax.set_title('Stabilitas K-Fold Semua Model\n(error bar = std across 5 folds)', fontsize=12)
ax.set_ylim(0.72, 0.92)
legend_patches = [mpatches.Patch(color=v, label=k) for k, v in CAT_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=9)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_kfold_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_kfold_stability.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 5: Radar Chart — Proposed vs Best Baseline per kategori
# ══════════════════════════════════════════════════════════════════════════════
from matplotlib.patches import FancyArrowPatch

radar_metrics = ['test_f1_macro','test_accuracy','test_precision','test_recall','test_f1_weighted']
radar_labels  = ['F1-Macro','Accuracy','Precision','Recall','F1-Weighted']
N             = len(radar_metrics)
angles        = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles       += angles[:1]  # tutup lingkaran

# Ambil model yang dibandingkan: Proposed, Best DL, Best ML
proposed_row  = df_all[df_all['category']=='Proposed'].iloc[0]
dl_row        = df_all[df_all['category']=='DL Baseline'].iloc[0]
ml_row        = df_all[df_all['category']=='ML Baseline'].iloc[0]

def get_radar_vals(row):
    vals = [getattr(row, m) for m in radar_metrics]
    return vals + vals[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for row, color, label in [
    (proposed_row, '#1D9E75', f'IndoBERT+CNN (Proposed)'),
    (dl_row,       '#EF9F27', f'IndoBERT-only (DL Baseline)'),
    (ml_row,       '#AFA9EC', f'{ml_row["model"]} (ML Baseline)'),
]:
    vals = get_radar_vals(row)
    ax.plot(angles, vals, 'o-', color=color, linewidth=2, label=label)
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=11)
ax.set_ylim(0.78, 0.90)
ax.set_yticks([0.78, 0.82, 0.86, 0.90])
ax.set_yticklabels(['0.78','0.82','0.86','0.90'], fontsize=8)
ax.set_title('Radar Chart — Proposed vs Best Baselines\n(Test Set)', fontsize=12, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_radar.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GRAFIK 6: Delta F1-Macro — Proposed vs Semua Baseline
# ══════════════════════════════════════════════════════════════════════════════
proposed_f1_val = df_all[df_all['category']=='Proposed']['test_f1_macro'].values[0]
df_baseline     = df_all[df_all['category']!='Proposed'].copy()
df_baseline['delta_f1'] = proposed_f1_val - df_baseline['test_f1_macro']
df_baseline = df_baseline.sort_values('delta_f1', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
delta_colors = ['#1D9E75' if d > 0 else '#E24B4A' for d in df_baseline['delta_f1']]
bars = ax.barh(range(len(df_baseline)), df_baseline['delta_f1'],
               color=delta_colors, edgecolor='none', alpha=0.9, height=0.55)

for bar, val in zip(bars, df_baseline['delta_f1']):
    x_pos = val + 0.0005 if val >= 0 else val - 0.0005
    ha    = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', ha=ha, fontsize=9, fontweight='bold')

bl_labels = [m.replace(' (Proposed)','') for m in df_baseline['model']]
ax.set_yticks(range(len(df_baseline)))
ax.set_yticklabels(bl_labels, fontsize=10)
ax.axvline(x=0, color='#333333', linewidth=1)
ax.set_xlabel('Δ F1-Macro (Proposed − Baseline)', fontsize=11)
ax.set_title('Peningkatan F1-Macro Proposed Model vs Semua Baseline\n'
             '(hijau = proposed lebih baik, merah = baseline lebih baik)', fontsize=11)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/comparison_delta_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ comparison_delta_f1.png')

## 5. Analisis Error — Disagreement antar Model

In [ ]:
# Load semua test predictions
pred_files = {
    'SVM'             : f'{DIR_NB05}/svm_test_predictions.csv',
    'LogReg'          : f'{DIR_NB05}/logisticregression_test_predictions.csv',
    'RF'              : f'{DIR_NB05}/randomforest_test_predictions.csv',
    'IndoBERT-only'   : f'{DIR_NB04}/bert_only_test_predictions.csv',
    'IndoBERT+CNN'    : f'{DIR_NB03}/test_predictions.csv',
}

df_base = None
for model_key, fpath in pred_files.items():
    if not os.path.exists(fpath):
        print(f'⚠️  File tidak ditemukan: {fpath}')
        continue
    df_pred = pd.read_csv(fpath)[['label','predicted']].rename(
        columns={'predicted': f'pred_{model_key}'})
    if df_base is None:
        df_base = df_pred
    else:
        df_base[f'pred_{model_key}'] = df_pred[f'pred_{model_key}']

if df_base is not None:
    # Hitung agreement rate per sample
    pred_cols = [c for c in df_base.columns if c.startswith('pred_')]
    df_base['n_correct'] = df_base.apply(
        lambda r: sum(r[c] == r['label'] for c in pred_cols), axis=1)
    df_base['all_agree']    = df_base.apply(
        lambda r: len(set(r[c] for c in pred_cols)) == 1, axis=1)
    df_base['all_correct']  = df_base['n_correct'] == len(pred_cols)
    df_base['all_wrong']    = df_base['n_correct'] == 0
    df_base['proposed_right_others_wrong'] = (
        (df_base['pred_IndoBERT+CNN'] == df_base['label']) &
        (df_base['n_correct'] == 1)
    )

    n_total = len(df_base)
    print(f'{"="*60}')
    print(f' ANALISIS ERROR — Test Set ({n_total:,} data)')
    print(f'{"="*60}')
    print(f'  Semua model benar         : {df_base["all_correct"].sum():,} ({df_base["all_correct"].mean()*100:.1f}%)')
    print(f'  Semua model salah         : {df_base["all_wrong"].sum():,} ({df_base["all_wrong"].mean()*100:.1f}%)')
    print(f'  Semua model sepakat       : {df_base["all_agree"].sum():,} ({df_base["all_agree"].mean()*100:.1f}%)')
    print(f'  Proposed benar, lain salah: {df_base["proposed_right_others_wrong"].sum():,} '
          f'({df_base["proposed_right_others_wrong"].mean()*100:.1f}%)')
    print(f'{"─"*60}')
    print(f'  Distribusi jumlah model yang benar per sample:')
    for n, cnt in df_base['n_correct'].value_counts().sort_index().items():
        print(f'    {n}/{len(pred_cols)} model benar: {cnt:,} sample ({cnt/n_total*100:.1f}%)')

    # Simpan
    df_base.to_csv(f'{OUTPUT_DIR}/error_analysis_all_models.csv',
                   index=False, encoding='utf-8-sig')
    print(f'\n  ✔ error_analysis_all_models.csv disimpan')
else:
    print('⚠️  Tidak ada file prediksi yang bisa dimuat.')

## 6. Ringkasan Final & Output

In [ ]:
# ── Simpan tabel ringkasan untuk laporan ──────────────────────────────────────
cols_report = ['model','category','test_f1_macro','test_f1_weighted',
               'test_accuracy','test_precision','test_recall',
               'f1_positive','f1_negative','f1_neutral',
               'kfold_f1_macro_mean','kfold_f1_macro_std']
df_report = df_all[cols_report].copy()
df_report.to_csv(f'{OUTPUT_DIR}/model_comparison_report.csv', index=False)

print(f'{"="*65}')
print(f' RINGKASAN FINAL — KOMPARASI SEMUA MODEL')
print(f'{"="*65}')
print(f'\n Test Set F1-Macro (metrik utama):')
for row in df_all.sort_values('test_f1_macro', ascending=False).itertuples():
    marker = ' ← proposed' if row.category == 'Proposed' else ''
    print(f'  {row.model:<32}: {row.test_f1_macro:.4f}{marker}')

# Temuan utama
proposed_val = df_all[df_all['category']=='Proposed']['test_f1_macro'].values[0]
best_dl_val  = df_all[df_all['category']=='DL Baseline']['test_f1_macro'].values[0]
best_ml_val  = df_all[df_all['category']=='ML Baseline']['test_f1_macro'].values[0]

print(f'\n Temuan Utama:')
print(f'  1. Proposed (IndoBERT+CNN) vs Best ML (SVM)  : Δ = {proposed_val-best_ml_val:+.4f}')
print(f'  2. Proposed (IndoBERT+CNN) vs Best DL (BERT) : Δ = {proposed_val-best_dl_val:+.4f}')
print(f'  3. Semua DL model (BERT-based) > semua ML model.')
if proposed_val < best_dl_val:
    print(f'  4. IndoBERT-only sedikit lebih baik dari IndoBERT+CNN pada dataset ini.')
    print(f'     Interpretasi: self-attention BERT sudah cukup untuk teks pendek Twitter.')
else:
    print(f'  4. IndoBERT+CNN berhasil melampaui IndoBERT-only.')

print(f'\n{"="*65}')
print(f' OUTPUT NOTEBOOK 06')
print(f'{"="*65}')
outputs = [
    'model_comparison_full.csv          → Dataset lengkap semua metrik',
    'model_comparison_report.csv        → Tabel siap laporan Bab 4',
    'error_analysis_all_models.csv      → Analisis disagreement per sample',
    'comparison_f1_macro.png            → Bar chart F1-Macro ranking',
    'comparison_all_metrics.png         → Grouped bar semua metrik',
    'comparison_perclass_f1.png         → Per-class F1 horizontal bar',
    'comparison_kfold_stability.png     → K-Fold stability dengan error bar',
    'comparison_radar.png               → Radar chart proposed vs baselines',
    'comparison_delta_f1.png            → Delta F1 proposed vs semua baseline',
]
for o in outputs:
    fname = o.split(' ')[0]
    exists = '✔' if os.path.exists(f'{OUTPUT_DIR}/{fname}') else '○'
    print(f'  {exists} {o}')

print(f'\n Pipeline research selesai! 🎉')
print(f' Notebook yang telah diselesaikan:')
for nb in ['01 — Stratified Sampling','02 — Preprocessing & Labeling',
           '03 — IndoBERT+CNN Training','04 — BERT-only Baseline',
           '05 — ML Baseline','06 — Model Comparison  ← selesai']:
    print(f'  ✅ {nb}')

---

## Catatan Metodologis untuk Laporan

### Validitas Komparasi
Komparasi antar model valid karena menggunakan **test set yang identik** (1.329 data, dipisah sebelum seluruh proses training) dan **metrik evaluasi yang sama** (F1-Macro sebagai metrik utama). Tidak ada data leakage dari test set ke proses training manapun.

### Interpretasi Hasil

**ML vs DL**: Semua model berbasis IndoBERT (DL) mengungguli model ML klasik secara signifikan, mengonfirmasi bahwa representasi kontekstual dari pre-trained Transformer lebih kaya dibanding TF-IDF untuk task analisis sentimen teks pendek.

**IndoBERT+CNN vs IndoBERT-only**: Selisih yang kecil antara keduanya mengindikasikan bahwa mekanisme self-attention 12-layer IndoBERT sudah mampu menangkap pola sentimen pada teks pendek Twitter secara efektif. Penambahan CNN 1D memberikan fitur n-gram lokal yang pada dataset teks pendek ini memberikan kontribusi marginal — konsisten dengan temuan Liu et al. (2019) bahwa arsitektur Transformer sudah sangat kompetitif untuk berbagai task NLP.

### Referensi
- Liu, Y., et al. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach. *arXiv:1907.11692*.
- Devlin et al. (2019). BERT. *NAACL-HLT 2019*.
- Kim, Y. (2014). CNN for Sentence Classification. *EMNLP 2014*.